# 09 · Consume Arrow one batch at a time

Export a bounded excerpt and process each Arrow record batch without building one large pandas table.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [NYC TLC trip records and taxi-zone lookup](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
This lesson uses the January 2024 yellow-taxi Parquet file. See the
[NYC Open Data terms of use](https://opendata.cityofnewyork.us/overview/#termsofuse)
and TLC's data dictionary and accuracy notice; do not treat public availability
as a warranty or apply the notebook's code license to the data.
The original Parquet download is roughly 50 MB and contains roughly three million
rows. The helper checks the pinned file digest. Queries use explicit result
limits; a small result does not imply a small source scan.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
import pyarrow.compute as pc
taxi = lab.taxi()
config = lab.catalog(sources=[{"id": "trips", "type": "parquet", "path": str(taxi)}])
EXPORT_ROWS = 100_000
# Optional larger export: set EXPORT_ROWS to 1_000_000 and review memory/disk limits first.
assert 1 <= EXPORT_ROWS <= 1_000_000
result = lab.query(config, f"""
    SELECT "PULocationID", "DOLocationID", total_amount::DECIMAL(18,2) AS total_amount
    FROM trips LIMIT {EXPORT_ROWS}
""", sources=["trips"], max_rows=EXPORT_ROWS, max_bytes=128 * 1024 * 1024)


## Reduce batches as they arrive from the exported stream

The CLI has completed before this file reader opens. The loop bounds the
Python-side retained result batches, not DuckDB's execution memory or the
export file size. Do not append every batch to a list if you want bounded
consumer memory.


In [ ]:
from decimal import Decimal
observed_rows = 0
batch_count = 0
largest_batch_bytes = 0
amount_total = Decimal(0)
for batch in result.batches():
    observed_rows += batch.num_rows
    batch_count += 1
    largest_batch_bytes = max(largest_batch_bytes, batch.nbytes)
    amount_total += pc.sum(batch.column("total_amount")).as_py() or Decimal(0)
assert observed_rows == EXPORT_ROWS == result.stats["rows"]
assert batch_count == result.stats["batches"] > 0
assert result.path.stat().st_size == result.stats["wire_bytes"]
print({"rows": observed_rows, "batches": batch_count,
       "largest_arrow_batch_bytes": largest_batch_bytes,
       "wire_bytes": result.path.stat().st_size,
       "sum_of_exported_amounts": str(amount_total)})


## Treat a result limit as an error, not a partial answer

This intentionally requests two rows with a one-row budget. A failed CLI
export must not publish a successful-looking Arrow file.


In [ ]:
rejected_path = lab.root / "must-not-exist.arrow"
rejected = lab.command([
    "query", "--config", str(config), "--sources", "trips",
    "--sql", "SELECT 1 AS value FROM trips LIMIT 2",
    "--max-rows", "1", "--out", str(rejected_path),
], check=False)
assert rejected.returncode != 0
assert not rejected_path.exists()
print("The oversized result was rejected without publishing an export.")


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
